In [ ]:
#| default_exp state

# Notebook state

> Explain which notebook work is current, changed, or likely affected by an upstream change.

In [ ]:
#| export
import ast, builtins, symtable
from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.foundation import nbskill_cell_metadata, cell_source, source_hash

In [ ]:
from tempfile import TemporaryDirectory

from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq

In [ ]:
#| exporti
def _state_scope_reads(table):
    reads = set()
    for child in table.get_children():
        reads.update(symbol.get_name() for symbol in child.get_symbols() if symbol.is_global() and symbol.is_referenced())
        reads.update(_state_scope_reads(child))
    return reads

In [ ]:
#| exporti
def _state_dynamic_reasons(tree):
    reasons = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {"eval", "exec", "globals", "locals", "vars", "setattr", "delattr"}:
            reasons.add("dynamic-namespace")
        targets = []
        if isinstance(node, (ast.Assign, ast.Delete)): targets = node.targets
        elif isinstance(node, (ast.AnnAssign, ast.AugAssign)): targets = [node.target]
        if any(isinstance(part, (ast.Attribute, ast.Subscript)) for target in targets for part in ast.walk(target)):
            reasons.add("in-place-mutation")
    return sorted(reasons)

In [ ]:
#| exporti
def _state_cell_facts(cell):
    source = cell_source(cell)
    try:
        tree = ast.parse(source)
        table = symtable.symtable(source, f"<cell {cell.id}>", "exec")
    except SyntaxError: return dict(defines=[], reads=[], uncertainty=["syntax-error"])
    symbols = table.get_symbols()
    defines = {symbol.get_name() for symbol in symbols if symbol.is_assigned() or symbol.is_imported() or symbol.is_namespace()}
    reads = {symbol.get_name() for symbol in symbols if symbol.is_referenced()} | _state_scope_reads(table)
    return dict(defines=sorted(defines), reads=sorted(reads - set(dir(builtins))), uncertainty=_state_dynamic_reasons(tree))

In [ ]:
#| exporti
def _state_direct(cell):
    current = source_hash(cell_source(cell), length=None)
    metadata = nbskill_cell_metadata(cell, create=False) or {}
    executed = metadata.get("executed_hash", "")
    has_error = any(output.get("output_type") == "error" for output in cell.get("outputs", []))
    if executed == current: state = "error" if has_error else "clean"
    elif executed: state = "dirty"
    else: state = "unknown"
    return state, current, executed

In [ ]:
#| exporti
def _state_dependencies(records):
    providers, dependencies = {}, {record["cell_id"]: {} for record in records}
    for record in records:
        target = record["cell_id"]
        for name in record["reads"]:
            if source := providers.get(name): dependencies[target].setdefault(source, []).append(name)
        for name in record["defines"]: providers[name] = target
    return dependencies

In [ ]:
#| exporti
def _state_propagate(records, dependencies):
    by_id = {record["cell_id"]: record for record in records}
    changed = True
    while changed:
        changed = False
        for record in records:
            if record["state"] in {"dirty", "error"}: continue
            roots = set()
            for source_id in dependencies[record["cell_id"]]:
                source = by_id[source_id]
                if source["state"] in {"dirty", "error"}: roots.add(source_id)
                elif source["state"] == "stale": roots.update(source["stale_from"])
            if roots and (record["state"] != "stale" or roots != set(record["stale_from"])):
                record.update(state="stale", confidence="likely", stale_from=sorted(roots))
                changed = True

In [ ]:
#| exporti
def _state_records(nb):
    records = []
    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != "code" or not cell_source(cell).strip(): continue
        facts = _state_cell_facts(cell)
        state, current, executed = _state_direct(cell)
        record = dict(cell_id=str(cell.id), cell_idx=idx, state=state, **facts)
        record.update(confidence="high" if state != "unknown" else "low")
        record.update(source_hash=current, executed_hash=executed, stale_from=[], depends_on=[])
        records.append(record)
    return records

In [ ]:
#| exporti
def _state_edges(dependencies):
    return [dict(source=source, target=target, symbols=sorted(symbols), confidence="likely")
        for target, sources in dependencies.items() for source, symbols in sources.items()]

In [ ]:
#| export
def notebook_state(path):
    "Return evidence-backed execution state and likely downstream impact for one notebook."
    path = Path(path)
    records = _state_records(read_nb(path))
    dependencies = _state_dependencies(records)
    for record in records: record["depends_on"] = sorted(dependencies[record["cell_id"]])
    _state_propagate(records, dependencies)
    states = ("clean", "dirty", "stale", "error", "unknown")
    counts = {state: sum(record["state"] == state for record in records) for state in states}
    limits = ["Static name dependencies are conservative.", "Dynamic mutation and namespace access can hide impact."]
    return dict(path=str(path), cells=records, dependencies=_state_edges(dependencies), counts=counts, limits=limits)

## Evidence-backed notebook state

`notebook_state` compares every code cell with the exact source hash recorded by `exec_nb`. A matching hash is `clean`, a changed hash is `dirty`, and missing execution evidence is `unknown`. Static name dependencies propagate a changed upstream cell as `stale` without claiming that Python is fully reactive.

Each dependency is marked `likely`. Calls such as `globals()` and assignment through attributes or subscripts appear under `uncertainty`, because AST and symbol-table analysis cannot prove the complete effect of dynamic Python.

A fresh notebook has three cells with exact execution fingerprints and one untracked dynamic cell. The report distinguishes `clean` from `unknown` without guessing.

In [ ]:
state_notebook = Path("15_state.ipynb")
if not state_notebook.exists(): state_notebook = Path("nbs") / state_notebook
notebook_state(state_notebook)["counts"]

In [ ]:
state_folder = TemporaryDirectory()
state_path = Path(state_folder.name) / "state.ipynb"
state_nb = new_nb([mk_cell("raw = 1"), mk_cell("clean = raw + 1"),
    mk_cell("result = clean * 2"), mk_cell('globals()["candidate"] = result')])
for state_cell in state_nb.cells[:3]: nbskill_cell_metadata(state_cell)["executed_hash"] = source_hash(cell_source(state_cell), length=None)
write_nb(state_nb, state_path)
initial_state = notebook_state(state_path)
test_eq(notebook_state(state_path)["counts"]["clean"], 3)
test_eq([cell["state"] for cell in initial_state["cells"]], ["clean", "clean", "clean", "unknown"])
test_eq(initial_state["cells"][3]["uncertainty"], ["dynamic-namespace", "in-place-mutation"])
initial_state

Changing only the first cell makes it `dirty`. Static name dependencies mark the other three cells `stale`, while `stale_from` keeps the changed root visible.

In [ ]:
changed_nb = read_nb(state_path)
changed_nb.cells[0].source = "raw = 2"
write_nb(changed_nb, state_path)
changed_state = notebook_state(state_path)
test_eq([cell["state"] for cell in changed_state["cells"]], ["dirty", "stale", "stale", "stale"])
test_eq(changed_state["cells"][1]["stale_from"], [changed_nb.cells[0].id])
state_folder.cleanup()
changed_state